# 04 — Tuning

- Tune the prior `(L, ALPHA)` and XGBoost hyperparameters on validation only.
- Test seasons remain untouched.

## 1. Imports & Config

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from scipy.stats import poisson
from collections import defaultdict
from pathlib import Path
import math, json, random, datetime, itertools

pd.set_option('display.max_columns', None)

RANDOM_SEED = 2026
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATA_PATH = Path('../Outputs/Cleaning/df_cleaned.csv')
FEAT_DIR  = Path('../Outputs/Features')
TUNE_DIR  = Path('../Outputs/Tuning'); TUNE_DIR.mkdir(parents=True, exist_ok=True)

MAX_GOALS_PER_TEAM = 10
TOTAL_GOALS_CAP    = 8
GOAL_BUCKETS       = list(range(TOTAL_GOALS_CAP)) + ['8+']
LEAGUE_RANK        = {'Premier League': 2, 'Championship': 1, 'League One': 0}

# Set True to delete intermediate grid CSVs and force a full re-run.
# True for the final run (extended grids invalidate cached winners); flip back to False afterwards.
CLEAR_CACHE = True

In [2]:
_CACHE_FILES = [
    TUNE_DIR / 'la_grid_search.csv',
    TUNE_DIR / 'la_grid_recheck.csv',
    TUNE_DIR / 'xgb_pairwise_grid_search.csv',
]
if CLEAR_CACHE:
    for _f in _CACHE_FILES:
        if _f.exists():
            _f.unlink()
            print(f'Cleared: {_f.name}')
    print('Cache cleared.')
else:
    print('CLEAR_CACHE=False — existing intermediate CSVs will be reused.')

Cache cleared.


## 2. Load Notebook 03 Outputs

In [3]:
sel          = json.loads((FEAT_DIR / "selected_features.json").read_text())
tuning_start = json.loads((FEAT_DIR / "tuning_start.json").read_text())
prev_cfg     = json.loads((FEAT_DIR / "run_config.json").read_text())

SELECTED_FEATS = sel["selected_features"]
NB03_LOGLOSS   = sel["selected_logloss"]
PREV_BASELINE  = sel["baseline_logloss"]

L03     = int(tuning_start["starting_L"])
ALPHA03 = float(tuning_start["starting_ALPHA"])
XGB03   = dict(tuning_start["starting_xgb_params"])
XGB03["random_state"] = RANDOM_SEED

TEST_SEASON = prev_cfg["test_season_cutoff"]

assert len(SELECTED_FEATS) > 0
assert len(set(SELECTED_FEATS)) == len(SELECTED_FEATS)
for f in SELECTED_FEATS:
    assert f in sel["full_features"], f"unknown feature {f!r}"

print(f"Selected features ({len(SELECTED_FEATS)}): {SELECTED_FEATS}")
print(f"03 reference logloss : selected={NB03_LOGLOSS:.6f}  baseline={PREV_BASELINE:.6f}")
print(f"03 starting (L, ALPHA): ({L03}, {ALPHA03})")
print(f"03 starting XGB params: {XGB03}")
print(f"Test season cutoff    : {TEST_SEASON}")

Selected features (5): ['team_avg_gls_f_v', 'team_avg_xg_f', 'team_avg_xg_c', 'opp_avg_xg_c', 'game_week_normalised']
03 reference logloss : selected=1.848385  baseline=1.848942
03 starting (L, ALPHA): (10, 0.0)
03 starting XGB params: {'objective': 'count:poisson', 'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 3, 'enable_categorical': True, 'random_state': 2026, 'verbosity': 0}
Test season cutoff    : 2024/2025


## 3. Rebuild Selected Features

Walk-forward builder using `SELECTED_FEATS`, shared `(L, ALPHA)`, and `L_V = ceil(0.5 * L)` for venue buffers (03's rule). No feature selection.

In [4]:
df_raw = pd.read_csv(DATA_PATH)
df_raw['date'] = pd.to_datetime(df_raw['date'])
df_raw = df_raw.sort_values('date').reset_index(drop=True)
print(f'Loaded cleaned: {df_raw.shape[0]:,} rows')

df = df_raw.copy()

home_map = df[['home_team', 'season', 'league']].rename(columns={'home_team': 'team'})
away_map = df[['away_team', 'season', 'league']].rename(columns={'away_team': 'team'})
team_season_league = (
    pd.concat([home_map, away_map])
      .drop_duplicates().sort_values(['team', 'season']).reset_index(drop=True)
)
team_season_league['prev_league'] = team_season_league.groupby('team')['league'].shift(1)
team_season_league['curr_rank']   = team_season_league['league'].map(LEAGUE_RANK)
team_season_league['prev_rank']   = team_season_league['prev_league'].map(LEAGUE_RANK)
team_season_league['promoted_flag']  = ((team_season_league['curr_rank'] > team_season_league['prev_rank']).fillna(False).astype(int))
team_season_league['relegated_flag'] = ((team_season_league['curr_rank'] < team_season_league['prev_rank']).fillna(False).astype(int))
flag_lookup = team_season_league.set_index(['team', 'season'])[['promoted_flag', 'relegated_flag']]

for side in ['home', 'away']:
    team_col = f'{side}_team'
    merged = df[[team_col, 'season']].merge(
        flag_lookup.rename(columns={'promoted_flag': f'{side}_promoted_flag',
                                    'relegated_flag': f'{side}_relegated_flag'}),
        left_on=[team_col, 'season'], right_index=True, how='left'
    )
    df[f'{side}_promoted_flag']  = merged[f'{side}_promoted_flag'].fillna(0).astype(int).values
    df[f'{side}_relegated_flag'] = merged[f'{side}_relegated_flag'].fillna(0).astype(int).values

xg_missing = df['home_xg'].isna() | df['away_xg'].isna()
df = df[~xg_missing].reset_index(drop=True)
df['total_goals'] = df['home_goals'] + df['away_goals']
print(f'After xG drop: {len(df):,} rows  ({df["date"].min().date()} \u2192 {df["date"].max().date()})')

Loaded cleaned: 11,239 rows
After xG drop: 10,687 rows  (2018-08-03 → 2026-02-16)


In [5]:
seasons_sorted = sorted(df['season'].unique())
l1_seeds = {}
for i, season in enumerate(seasons_sorted):
    if i == 0:
        continue
    prev = seasons_sorted[i - 1]
    sub  = df[(df['season'] == prev) & (df['league'] == 'League One')]
    if sub.empty:
        continue
    per_team_goals = (sub['home_goals'] + sub['away_goals']).mean() / 2.0
    per_team_xg    = (sub['home_xg']    + sub['away_xg']).mean()    / 2.0
    l1_seeds[season] = {'gls_f': per_team_goals, 'gls_c': per_team_goals,
                        'xg_f':  per_team_xg,    'xg_c':  per_team_xg}
print(f'L1 seeds defined for {len(l1_seeds)} seasons')

L1 seeds defined for 6 seasons


In [6]:
FEATURE_SPEC = {
    # Rolling prior features (team / opponent)
    'team_avg_gls_f':   {'scope': 'team', 'side': 'f', 'stat': 'gls', 'venue': 'all'},
    'team_avg_gls_c':   {'scope': 'team', 'side': 'c', 'stat': 'gls', 'venue': 'all'},
    'team_avg_xg_f':    {'scope': 'team', 'side': 'f', 'stat': 'xg',  'venue': 'all'},
    'team_avg_xg_c':    {'scope': 'team', 'side': 'c', 'stat': 'xg',  'venue': 'all'},
    'team_avg_gls_f_v': {'scope': 'team', 'side': 'f', 'stat': 'gls', 'venue': 'venue'},
    'team_avg_gls_c_v': {'scope': 'team', 'side': 'c', 'stat': 'gls', 'venue': 'venue'},
    'team_avg_xg_f_v':  {'scope': 'team', 'side': 'f', 'stat': 'xg',  'venue': 'venue'},
    'team_avg_xg_c_v':  {'scope': 'team', 'side': 'c', 'stat': 'xg',  'venue': 'venue'},
    'opp_avg_gls_f':    {'scope': 'opp',  'side': 'f', 'stat': 'gls', 'venue': 'all'},
    'opp_avg_gls_c':    {'scope': 'opp',  'side': 'c', 'stat': 'gls', 'venue': 'all'},
    'opp_avg_xg_f':     {'scope': 'opp',  'side': 'f', 'stat': 'xg',  'venue': 'all'},
    'opp_avg_xg_c':     {'scope': 'opp',  'side': 'c', 'stat': 'xg',  'venue': 'all'},
    'opp_avg_gls_f_v':  {'scope': 'opp',  'side': 'f', 'stat': 'gls', 'venue': 'venue'},
    'opp_avg_gls_c_v':  {'scope': 'opp',  'side': 'c', 'stat': 'gls', 'venue': 'venue'},
    'opp_avg_xg_f_v':   {'scope': 'opp',  'side': 'f', 'stat': 'xg',  'venue': 'venue'},
    'opp_avg_xg_c_v':   {'scope': 'opp',  'side': 'c', 'stat': 'xg',  'venue': 'venue'},
    # League venue-aware priors
    'league_avg_vgf':   {'scope': 'league', 'vgf': True},
    'league_avg_vgc':   {'scope': 'league', 'vgf': False},
    # Context (match-level, same for both teams)
    'is_crowds':             {'scope': 'context', 'col': 'is_crowds'},
    'game_week_normalised':  {'scope': 'context', 'col': 'game_week_normalised'},
    'league':                {'scope': 'context', 'col': 'league'},
    'is_home':               {'scope': 'context', 'col': None},
    # Movement flags (team-side-routed)
    'team_promoted_flag':  {'scope': 'movement', 'home_col': 'home_promoted_flag',  'away_col': 'away_promoted_flag'},
    'team_relegated_flag': {'scope': 'movement', 'home_col': 'home_relegated_flag', 'away_col': 'away_relegated_flag'},
    'opp_promoted_flag':   {'scope': 'movement', 'home_col': 'away_promoted_flag',  'away_col': 'home_promoted_flag'},
    'opp_relegated_flag':  {'scope': 'movement', 'home_col': 'away_relegated_flag', 'away_col': 'home_relegated_flag'},
}
for f in SELECTED_FEATS:
    assert f in FEATURE_SPEC, f'no spec for {f!r}'

max_gw = df.groupby(['league', 'season'])['game_week'].max().to_dict()

def _stat_key(stat, side):
    return f'{stat}_{side}'

def _weighted_mean(buf, window, alpha):
    recent = buf[-window:]
    k = len(recent)
    if k == 0:
        return np.nan
    w = np.exp(-alpha * (k - 1 - np.arange(k)))
    w = w / w.sum()
    return float(np.dot(w, recent))


In [7]:
def build_features_tuned(L, ALPHA):
    L = int(L)
    ALPHA = float(ALPHA)
    L_V = math.ceil(0.5 * L)

    # Rolling-prior stat keys needed for team/opp buffers only.
    rolling_feats = [f for f in SELECTED_FEATS if FEATURE_SPEC[f]['scope'] in ('team', 'opp')]
    needed_keys = sorted({_stat_key(FEATURE_SPEC[f]['stat'], FEATURE_SPEC[f]['side'])
                          for f in rolling_feats})

    # League buffer (only allocated if a league-scope feature is selected).
    _needs_league = any(FEATURE_SPEC[f]['scope'] == 'league' for f in SELECTED_FEATS)
    league_buffer = defaultdict(list)   # league -> [(season, gw, hg, ag), ...]

    team_gen  = defaultdict(lambda: defaultdict(list))
    team_home = defaultdict(lambda: defaultdict(list))
    team_away = defaultdict(lambda: defaultdict(list))
    team_home_count = defaultdict(int)
    team_away_count = defaultdict(int)

    feature_rows = []
    for match_id, row in df.iterrows():
        league = row['league']; season = row['season']; date = row['date']
        ht = row['home_team']; at = row['away_team']
        hg, ag   = row['home_goals'], row['away_goals']
        hxg, axg = row['home_xg'],    row['away_xg']
        gw       = row['game_week']
        match_tg = hg + ag

        seed = l1_seeds.get(season) if league == 'League One' else None
        nh_h, na_h = team_home_count[ht], team_away_count[ht]
        nh_a, na_a = team_home_count[at], team_away_count[at]

        # Build partial wide row with context/movement/league columns first.
        wide = {
            'match_id': match_id, 'date': date, 'season': season, 'league': league,
            'game_week': gw, 'home_team': ht, 'away_team': at,
            'home_goals': hg, 'away_goals': ag, 'total_goals': match_tg,
            'is_crowds':             row['is_crowds'],
            'game_week_normalised':  gw / max_gw.get((league, season), gw),
            'home_promoted_flag':    row['home_promoted_flag'],
            'away_promoted_flag':    row['away_promoted_flag'],
            'home_relegated_flag':   row['home_relegated_flag'],
            'away_relegated_flag':   row['away_relegated_flag'],
        }

        if _needs_league:
            buf_filtered = [(s, g, h, a) for (s, g, h, a) in league_buffer[league]
                            if (s, g) != (season, gw)][-L:]
            if buf_filtered:
                wide['league_avg_hg'] = float(np.mean([h for _, _, h, _ in buf_filtered]))
                wide['league_avg_ag'] = float(np.mean([a for _, _, _, a in buf_filtered]))
            else:
                wide['league_avg_hg'] = np.nan
                wide['league_avg_ag'] = np.nan

        def _read(side_is_home, feat_name):
            spec = FEATURE_SPEC[feat_name]
            sc = spec['scope']
            if sc in ('team', 'opp'):
                stat_key = _stat_key(spec['stat'], spec['side'])
                if sc == 'team':
                    t      = ht if side_is_home else at
                    nh, na = (nh_h, na_h) if side_is_home else (nh_a, na_a)
                    use_home_buf = side_is_home
                else:
                    t      = at if side_is_home else ht
                    nh, na = (nh_a, na_a) if side_is_home else (nh_h, na_h)
                    use_home_buf = not side_is_home
                if (nh < 1) or (na < 1):
                    return float(seed[stat_key]) if seed is not None else np.nan
                if spec['venue'] == 'all':
                    return _weighted_mean(team_gen[t][stat_key], L, ALPHA)
                buf = team_home[t][stat_key] if use_home_buf else team_away[t][stat_key]
                return _weighted_mean(buf, L_V, ALPHA)
            elif sc == 'league':
                hg_val = wide.get('league_avg_hg', np.nan)
                ag_val = wide.get('league_avg_ag', np.nan)
                return hg_val if (side_is_home == spec['vgf']) else ag_val
            elif sc == 'context':
                if spec['col'] is None:   # is_home
                    return int(side_is_home)
                return wide.get(spec['col'], np.nan)
            elif sc == 'movement':
                col = spec['home_col'] if side_is_home else spec['away_col']
                return float(wide.get(col, 0))
            return np.nan

        # Store rolling-prior features as home_/away_ columns in wide.
        for feat in rolling_feats:
            wide[f'home_{feat}'] = _read(True,  feat)
            wide[f'away_{feat}'] = _read(False, feat)
        feature_rows.append(wide)

        h_obs = {'gls_f': hg, 'gls_c': ag, 'xg_f': hxg, 'xg_c': axg}
        a_obs = {'gls_f': ag, 'gls_c': hg, 'xg_f': axg, 'xg_c': hxg}
        for k in needed_keys:
            team_gen[ht][k].append(h_obs[k])
            team_gen[at][k].append(a_obs[k])
            team_home[ht][k].append(h_obs[k])
            team_away[at][k].append(a_obs[k])
        team_home_count[ht] += 1
        team_away_count[at] += 1
        if _needs_league:
            league_buffer[league].append((season, gw, hg, ag))

    df_wide = pd.DataFrame(feature_rows)

    # NaN-drop on rolling-prior columns only (context/movement are always present).
    rolling_side_cols = [f'{p}_{f}' for p in ('home', 'away') for f in rolling_feats]
    if rolling_side_cols:
        df_wide = df_wide[~df_wide[rolling_side_cols].isna().any(axis=1)].reset_index(drop=True)

    season_dtype = pd.CategoricalDtype(categories=sorted(df_wide['season'].unique()), ordered=True)
    df_wide['season'] = df_wide['season'].astype(season_dtype)
    df_wide['league'] = df_wide['league'].astype('category')

    def _side_long(is_home):
        prefix     = 'home' if is_home else 'away'
        opp_prefix = 'away' if is_home else 'home'
        out = pd.DataFrame({
            'match_id': df_wide['match_id'].values,
            'date':     df_wide['date'].values,
            'season':   df_wide['season'].values,
            'league':   df_wide['league'].values,
            'team':     df_wide[f'{prefix}_team'].values,
            'opponent': df_wide[f'{opp_prefix}_team'].values,
            'is_home':  int(is_home),
            'team_goals':  df_wide[f'{prefix}_goals'].values,
            'total_goals': df_wide['total_goals'].values,
        })
        for f in SELECTED_FEATS:
            spec = FEATURE_SPEC[f]
            sc = spec['scope']
            if sc in ('team', 'opp'):
                out[f] = df_wide[f'{prefix}_{f}'].values
            elif sc == 'league':
                hg = df_wide['league_avg_hg'].values
                ag = df_wide['league_avg_ag'].values
                out[f] = hg if (is_home == spec['vgf']) else ag
            elif sc == 'context':
                if spec['col'] is None:   # is_home
                    out[f] = int(is_home)
                else:
                    out[f] = df_wide[spec['col']].values
            elif sc == 'movement':
                col = spec['home_col'] if is_home else spec['away_col']
                out[f] = df_wide[col].values
        return out

    df_long = pd.concat([_side_long(True), _side_long(False)], ignore_index=True)
    df_long['season'] = df_long['season'].astype(season_dtype)
    df_long['league'] = df_long['league'].astype('category')
    df_long = df_long.sort_values(['date', 'match_id', 'is_home'],
                                  ascending=[True, True, False]).reset_index(drop=True)
    return df_wide, df_long


In [8]:
df_wide_tuned, df_long_tuned = build_features_tuned(L03, ALPHA03)
print(f'Wide: {df_wide_tuned.shape[0]:,} matches  Long: {df_long_tuned.shape[0]:,} rows')

assert (df_long_tuned.groupby('match_id').size() == 2).all()
recon = df_long_tuned.groupby('match_id')['team_goals'].sum()
assert recon.equals(df_wide_tuned.set_index('match_id')['total_goals'].loc[recon.index])

df_dev_long = df_long_tuned[df_long_tuned['season'] < TEST_SEASON].copy()
dev_seasons = [str(s) for s in sorted(df_dev_long['season'].unique())]
val_seasons = sorted(df_dev_long['season'].unique())[1:]
print(f'Dev seasons        : {dev_seasons}')
print(f'Validation seasons : {[str(s) for s in val_seasons]}')
assert all(str(s) < TEST_SEASON for s in df_dev_long['season'].unique())

Wide: 10,610 matches  Long: 21,220 rows
Dev seasons        : ['2018/2019', '2019/2020', '2020/2021', '2021/2022', '2022/2023', '2023/2024']
Validation seasons : ['2019/2020', '2020/2021', '2021/2022', '2022/2023', '2023/2024']


## 4. Evaluation Helpers

In [9]:
def team_pmf(lmbda, max_goals=MAX_GOALS_PER_TEAM):
    ks = np.arange(max_goals + 1)
    p = poisson.pmf(ks, lmbda).astype(float)
    p[-1] += poisson.sf(max_goals, lmbda)
    return p

def scoreline_matrix(lambda_home, lambda_away, max_goals=MAX_GOALS_PER_TEAM):
    return np.outer(team_pmf(lambda_home, max_goals), team_pmf(lambda_away, max_goals))

def total_goals_pmf_from_matrix(M, cap=TOTAL_GOALS_CAP):
    n = M.shape[0]
    totals = np.zeros(2 * (n - 1) + 1)
    for i in range(n):
        for j in range(n):
            totals[i + j] += M[i, j]
    out = np.zeros(cap + 1)
    out[:cap] = totals[:cap]
    out[cap]  = totals[cap:].sum()
    return out

def total_goals_pmf_from_lambda(lmbda, cap=TOTAL_GOALS_CAP):
    ks  = np.arange(cap)
    out = np.zeros(cap + 1)
    out[:cap] = poisson.pmf(ks, lmbda)
    out[cap]  = poisson.sf(cap - 1, lmbda)
    return out

def _check_pmf(pmf):
    assert len(pmf) == len(GOAL_BUCKETS)
    assert np.isclose(pmf.sum(), 1.0, atol=1e-6)
    return pmf

def total_goals_logloss(pmf, actual_total, cap=TOTAL_GOALS_CAP, eps=1e-15):
    return -np.log(max(pmf[min(int(actual_total), cap)], eps))

def bucket_actual(actual_total, cap=TOTAL_GOALS_CAP):
    return min(int(actual_total), cap)

def append_row(path, row_dict):
    pd.DataFrame([row_dict]).to_csv(
        path, mode='a', header=not (path.exists() and path.stat().st_size > 0),
        index=False)

def run_xgb_selected(dev_long, val_seasons, params, feats=None, label='XGB'):
    if feats is None:
        feats = SELECTED_FEATS
    recs = []
    for vs in val_seasons:
        train = dev_long[dev_long['season'] < vs]
        val   = dev_long[dev_long['season'] == vs]
        if train.empty or val.empty:
            continue
        valid_leagues = set(train['league'].unique())
        val = val[val['league'].isin(valid_leagues)].copy()
        if val.empty:
            continue
        model = xgb.XGBRegressor(**params)
        model.fit(train[feats], train['team_goals'])
        val['lam'] = np.clip(model.predict(val[feats]), 1e-8, None)
        for mid, grp in val.groupby('match_id'):
            home = grp[grp['is_home'] == 1]; away = grp[grp['is_home'] == 0]
            if len(home) != 1 or len(away) != 1:
                continue
            lh = float(home['lam'].iloc[0]); la = float(away['lam'].iloc[0])
            pmf = _check_pmf(total_goals_pmf_from_matrix(scoreline_matrix(lh, la)))
            recs.append({
                'model': label, 'val_season': vs, 'match_id': mid,
                'league': home['league'].iloc[0],
                'total_goals': int(home['total_goals'].iloc[0]),
                'expected': lh + la, 'pmf': pmf,
                'lambda_home': lh, 'lambda_away': la,
            })
    return pd.DataFrame(recs)

def cv_logloss_selected(dev_long, val_seasons, params, feats=None):
    preds = run_xgb_selected(dev_long, val_seasons, params=params, feats=feats)
    if preds.empty:
        return float('nan'), float('nan'), {}
    ll   = preds.apply(lambda r: total_goals_logloss(r['pmf'], r['total_goals']), axis=1)
    fold = ll.groupby(preds['val_season'], observed=True).mean()
    return float(fold.mean()), float(fold.std()), {str(k): float(v) for k, v in fold.items()}

_check_pmf(total_goals_pmf_from_lambda(2.6))
_check_pmf(total_goals_pmf_from_matrix(scoreline_matrix(1.4, 1.1)))
print('PMF helpers self-checked.')

PMF helpers self-checked.


## 5. Starting Model Sanity Check

Confirm that 04 reproduces the 03 selected-model result before tuning begins.

In [10]:
def _spot_check_leakage():
    # Find a team-scope, all-venue rolling prior to verify.
    rolling_all = [f for f in SELECTED_FEATS
                   if FEATURE_SPEC[f]['scope'] == 'team' and FEATURE_SPEC[f].get('venue') == 'all']
    if not rolling_all:
        print('Leakage spot check: no team+all-venue feature selected — skipped.')
        return
    feat = rolling_all[0]
    spec = FEATURE_SPEC[feat]
    candidate = df_wide_tuned.iloc[len(df_wide_tuned) // 2]
    mid = candidate['match_id']; team = candidate['home_team']; date = candidate['date']
    prior_rows = df[(df['date'] < date) & ((df['home_team'] == team) | (df['away_team'] == team))]
    prior_rows = prior_rows.sort_values('date')
    obs = []
    for _, r in prior_rows.iterrows():
        is_home = (r['home_team'] == team)
        gf = r['home_goals'] if is_home else r['away_goals']
        gc = r['away_goals'] if is_home else r['home_goals']
        xf = r['home_xg']    if is_home else r['away_xg']
        xc = r['away_xg']    if is_home else r['home_xg']
        obs.append({'gls_f': gf, 'gls_c': gc, 'xg_f': xf, 'xg_c': xc}[_stat_key(spec['stat'], spec['side'])])
    expected = _weighted_mean(obs, L03, ALPHA03)
    actual   = candidate[f'home_{feat}']
    ok = np.isclose(expected, actual, atol=1e-9, equal_nan=True)
    print(f'Leakage spot check ({feat}, match {mid}): expected={expected:.6f}  actual={actual:.6f}  -> {"OK" if ok else "MISMATCH"}')
    assert ok, 'leakage spot-check failed'

_spot_check_leakage()


Leakage spot check (team_avg_xg_f, match 5382): expected=0.697350  actual=0.697350  -> OK


In [11]:
start_ll, start_std, _ = cv_logloss_selected(df_dev_long, val_seasons, XGB03)
print(f'03 saved logloss : {NB03_LOGLOSS:.6f}')
print(f'04 reproduction  : {start_ll:.6f}')
gap = start_ll - NB03_LOGLOSS
if abs(gap) < 0.002:
    print(f'OK — within \u00b10.002 (gap={gap:+.6f}).')
else:
    print(f'WARNING — gap {gap:+.6f} exceeds \u00b10.002. Investigate before tuning.')

03 saved logloss : 1.848385
04 reproduction  : 1.848385
OK — within ±0.002 (gap=+0.000000).


## 6. Shared L / ALPHA Tuning

In [12]:
L_GRID     = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
ALPHA_GRID = [0.0, 0.02, 0.05, 0.1, 0.15, 0.2, 0.25, 0.35, 0.5]
LA_GRID_CSV = TUNE_DIR / 'la_grid_search.csv'

def _grid_done_pairs(path):
    if path.exists() and path.stat().st_size > 0:
        prev = pd.read_csv(path)
        return set(zip(prev['L'].astype(int), prev['ALPHA'].astype(float)))
    return set()

done  = _grid_done_pairs(LA_GRID_CSV)
total = len(L_GRID) * len(ALPHA_GRID)
i = 0
for L_try in L_GRID:
    for A_try in ALPHA_GRID:
        i += 1
        if (int(L_try), float(A_try)) in done:
            continue
        _, dfl_t = build_features_tuned(L_try, A_try)
        dfl_dev  = dfl_t[dfl_t['season'] < TEST_SEASON]
        ll_mean, ll_std, fold_d = cv_logloss_selected(dfl_dev, val_seasons, XGB03)
        append_row(LA_GRID_CSV, {
            'L': int(L_try), 'ALPHA': float(A_try),
            'fold_loglosses_json': json.dumps(fold_d),
            'logloss_mean': ll_mean, 'logloss_std': ll_std,
        })
        print(f'  ({i:2d}/{total}) L={L_try:>2}, ALPHA={A_try:.2f} -> ll={ll_mean:.6f} (\u00b1{ll_std:.6f})')

  ( 1/90) L= 5, ALPHA=0.00 -> ll=1.850095 (±0.006566)
  ( 2/90) L= 5, ALPHA=0.02 -> ll=1.850054 (±0.006684)
  ( 3/90) L= 5, ALPHA=0.05 -> ll=1.851025 (±0.005437)
  ( 4/90) L= 5, ALPHA=0.10 -> ll=1.851237 (±0.003974)
  ( 5/90) L= 5, ALPHA=0.15 -> ll=1.851797 (±0.003908)
  ( 6/90) L= 5, ALPHA=0.20 -> ll=1.850228 (±0.006617)
  ( 7/90) L= 5, ALPHA=0.25 -> ll=1.851655 (±0.004491)
  ( 8/90) L= 5, ALPHA=0.35 -> ll=1.850749 (±0.005704)
  ( 9/90) L= 5, ALPHA=0.50 -> ll=1.852156 (±0.004690)
  (10/90) L=10, ALPHA=0.00 -> ll=1.848385 (±0.005837)
  (11/90) L=10, ALPHA=0.02 -> ll=1.847199 (±0.007017)
  (12/90) L=10, ALPHA=0.05 -> ll=1.847967 (±0.005426)
  (13/90) L=10, ALPHA=0.10 -> ll=1.847895 (±0.006622)
  (14/90) L=10, ALPHA=0.15 -> ll=1.847156 (±0.006020)
  (15/90) L=10, ALPHA=0.20 -> ll=1.848857 (±0.005153)
  (16/90) L=10, ALPHA=0.25 -> ll=1.848166 (±0.006829)
  (17/90) L=10, ALPHA=0.35 -> ll=1.850140 (±0.005611)
  (18/90) L=10, ALPHA=0.50 -> ll=1.852199 (±0.005292)
  (19/90) L=15, ALPHA=0.00 -

In [13]:
grid   = pd.read_csv(LA_GRID_CSV)
chosen = grid.sort_values(['logloss_mean', 'logloss_std']).iloc[0]
best_L, best_ALPHA = int(chosen['L']), float(chosen['ALPHA'])
best_prior_logloss = float(chosen['logloss_mean'])
print(f'Best (L, ALPHA): L={best_L}, ALPHA={best_ALPHA}  ll={best_prior_logloss:.6f} (\u00b1{chosen["logloss_std"]:.6f})')

df_wide_best, df_long_best = build_features_tuned(best_L, best_ALPHA)
df_long_dev_best = df_long_best[df_long_best['season'] < TEST_SEASON].copy()

Best (L, ALPHA): L=40, ALPHA=0.05  ll=1.843393 (±0.007584)


## 7. XGBoost Pairwise Tuning (from 03_best)

4-block pairwise grid descent starting from 03_best params only.

In [14]:
XGB_PAIRWISE_CSV = TUNE_DIR / 'xgb_pairwise_grid_search.csv'

PAIR_BLOCKS = [
    ('n_estimators+learning_rate', [
        ('n_estimators',  [100, 200, 400, 600,800,1000,1200]),
        ('learning_rate', [0.01, 0.02, 0.05, 0.1,0.2]),
    ]),
    ('max_depth+min_child_weight', [
        ('max_depth',        [2, 3, 4, 5]),
        ('min_child_weight', [1,5, 10, 20, 30]),
    ]),
    ('subsample+colsample_bytree', [
        ('subsample',        [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]),
        ('colsample_bytree', [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]),
    ]),
    ('reg_alpha+reg_lambda', [
        ('reg_alpha',  [0, 1, 5, 10, 20]),
        ('reg_lambda', [0.1, 0.5, 1.0, 5.0]),
    ]),
]

def _done_pair_keys(path):
    if path.exists() and path.stat().st_size > 0:
        prev = pd.read_csv(path)
        return set(zip(prev['start'], prev['block'], prev['params_json']))
    return set()

def _run_descent(start_label, init_params, csv_path):
    current   = dict(init_params)
    done_keys = _done_pair_keys(csv_path)
    for block_name, axes in PAIR_BLOCKS:
        print(f'  [{start_label}] block: {block_name}')
        (k1, v1s), (k2, v2s) = axes
        for v1 in v1s:
            for v2 in v2s:
                trial = dict(current)
                trial[k1] = int(v1) if isinstance(v1, int) else float(v1)
                trial[k2] = int(v2) if isinstance(v2, int) else float(v2)
                params_json = json.dumps(trial, sort_keys=True)
                if (start_label, block_name, params_json) in done_keys:
                    continue
                ll_mean, ll_std, fold_d = cv_logloss_selected(df_long_dev_best, val_seasons, trial)
                append_row(csv_path, {
                    'start': start_label, 'block': block_name,
                    'params_json': params_json,
                    'fold_loglosses_json': json.dumps(fold_d),
                    'logloss_mean': ll_mean, 'logloss_std': ll_std,
                })
        sub    = pd.read_csv(csv_path)
        sub    = sub[(sub['start'] == start_label) & (sub['block'] == block_name)]
        winner = sub.sort_values(['logloss_mean', 'logloss_std']).iloc[0]
        current = json.loads(winner['params_json'])
        print(f'    -> winner: ll={winner["logloss_mean"]:.6f}  {k1}={current[k1]}  {k2}={current[k2]}')
    return current

In [15]:
best_xgb_params = _run_descent('03_best', XGB03, XGB_PAIRWISE_CSV)

sub = pd.read_csv(XGB_PAIRWISE_CSV)
sub = sub[(sub['start'] == '03_best') & (sub['block'] == PAIR_BLOCKS[-1][0])]
final_descent = sub.sort_values(['logloss_mean', 'logloss_std']).iloc[0]
print(f'\n03_best end-of-descent logloss: {final_descent["logloss_mean"]:.6f} (\u00b1{final_descent["logloss_std"]:.6f})')
print(json.dumps(best_xgb_params, indent=2))

  [03_best] block: n_estimators+learning_rate
    -> winner: ll=1.841824  n_estimators=400  learning_rate=0.01
  [03_best] block: max_depth+min_child_weight
    -> winner: ll=1.841816  max_depth=3  min_child_weight=5
  [03_best] block: subsample+colsample_bytree
    -> winner: ll=1.840392  subsample=0.5  colsample_bytree=0.6
  [03_best] block: reg_alpha+reg_lambda
    -> winner: ll=1.840344  reg_alpha=1  reg_lambda=0.1

03_best end-of-descent logloss: 1.840344 (±0.010204)
{
  "colsample_bytree": 0.6,
  "enable_categorical": true,
  "learning_rate": 0.01,
  "max_depth": 3,
  "min_child_weight": 5,
  "n_estimators": 400,
  "objective": "count:poisson",
  "random_state": 2026,
  "reg_alpha": 1,
  "reg_lambda": 0.1,
  "subsample": 0.5,
  "verbosity": 0
}


## 8. Stability Re-check

Re-run the L/ALPHA grid at the tuned XGB params. If the winner moves, update and stop — no further tuning.

In [16]:
LA_RECHECK_CSV = TUNE_DIR / 'la_grid_recheck.csv'

done_rc = _grid_done_pairs(LA_RECHECK_CSV)
for L_try in L_GRID:
    for A_try in ALPHA_GRID:
        if (int(L_try), float(A_try)) in done_rc:
            continue
        _, dfl_t = build_features_tuned(L_try, A_try)
        dfl_dev  = dfl_t[dfl_t['season'] < TEST_SEASON]
        ll_mean, ll_std, fold_d = cv_logloss_selected(dfl_dev, val_seasons, best_xgb_params)
        append_row(LA_RECHECK_CSV, {
            'L': int(L_try), 'ALPHA': float(A_try),
            'fold_loglosses_json': json.dumps(fold_d),
            'logloss_mean': ll_mean, 'logloss_std': ll_std,
        })

rc        = pd.read_csv(LA_RECHECK_CSV)
rc_chosen = rc.sort_values(['logloss_mean', 'logloss_std']).iloc[0]
best_L_recheck    = int(rc_chosen['L'])
best_ALPHA_recheck = float(rc_chosen['ALPHA'])
print(f'Recheck best: L={best_L_recheck}, ALPHA={best_ALPHA_recheck}  ll={rc_chosen["logloss_mean"]:.6f} (\u00b1{rc_chosen["logloss_std"]:.6f})')

stability_moved = (best_L_recheck, best_ALPHA_recheck) != (best_L, best_ALPHA)
if not stability_moved:
    print('Stable.')
else:
    print(f'Drift: ({best_L}, {best_ALPHA}) \u2192 ({best_L_recheck}, {best_ALPHA_recheck})')
    best_L, best_ALPHA = best_L_recheck, best_ALPHA_recheck
    df_wide_best, df_long_best = build_features_tuned(best_L, best_ALPHA)
    df_long_dev_best = df_long_best[df_long_best['season'] < TEST_SEASON].copy()
    print(f'Updated to L={best_L}, ALPHA={best_ALPHA}.')

Recheck best: L=40, ALPHA=0.05  ll=1.840344 (±0.010204)
Stable.


## 9. Final Validation Comparison

In [17]:
def run_baseline_selected(dev_wide, val_seasons):
    recs = []
    for vs in val_seasons:
        train = dev_wide[dev_wide['season'] < vs]
        val   = dev_wide[dev_wide['season'] == vs]
        if train.empty or val.empty:
            continue
        lambda_map = train.groupby('league', observed=True)['total_goals'].mean().to_dict()
        val = val[val['league'].isin(lambda_map.keys())]
        for _, r in val.iterrows():
            lam = float(lambda_map[r['league']])
            pmf = _check_pmf(total_goals_pmf_from_lambda(lam))
            recs.append({'model': 'Baseline', 'val_season': vs, 'match_id': r['match_id'],
                         'league': r['league'], 'total_goals': int(r['total_goals']),
                         'expected': lam, 'pmf': pmf,
                         'lambda_home': np.nan, 'lambda_away': np.nan})
    return pd.DataFrame(recs)

def _metrics_block(preds, model_name, n_features):
    ll   = preds.apply(lambda r: total_goals_logloss(r['pmf'], r['total_goals']), axis=1)
    fold = ll.groupby(preds['val_season'], observed=True).mean()
    mae  = float((preds['expected'] - preds['total_goals']).abs().mean())
    rmse = float(np.sqrt(((preds['expected'] - preds['total_goals'])**2).mean()))
    top1 = float(preds.apply(lambda r: int(np.argmax(r['pmf'])) == bucket_actual(r['total_goals']), axis=1).mean())
    w1   = float(preds.apply(lambda r: abs(int(np.argmax(r['pmf'])) - bucket_actual(r['total_goals'])) <= 1, axis=1).mean())
    return {
        'model': model_name, 'n_features': n_features, 'n_matches': len(preds),
        'logloss_mean': float(fold.mean()), 'logloss_std': float(fold.std()),
        'mae': mae, 'rmse': rmse,
        'top1_accuracy': top1, 'within_1_accuracy': w1,
    }

In [18]:
df_long_start_dev = df_long_tuned[df_long_tuned['season'] < TEST_SEASON].copy()
df_wide_start_dev = df_wide_tuned[df_wide_tuned['season'] < TEST_SEASON].copy()

preds_baseline = run_baseline_selected(df_wide_start_dev, val_seasons)
preds_03_xgb   = run_xgb_selected(df_long_start_dev, val_seasons, XGB03,           label='03_Selected_XGB')
preds_04_xgb   = run_xgb_selected(df_long_dev_best,  val_seasons, best_xgb_params, label='04_Tuned_XGB')

common = (set(preds_baseline['match_id'])
        & set(preds_03_xgb['match_id'])
        & set(preds_04_xgb['match_id']))
print(f'Comparison set: {len(common):,} matches')

preds_baseline_cmp = preds_baseline[preds_baseline['match_id'].isin(common)]
preds_03_cmp       = preds_03_xgb  [preds_03_xgb  ['match_id'].isin(common)]
preds_04_cmp       = preds_04_xgb  [preds_04_xgb  ['match_id'].isin(common)]

rows = [
    _metrics_block(preds_baseline_cmp, 'Baseline',        0),
    _metrics_block(preds_03_cmp,       '03_Selected_XGB', len(SELECTED_FEATS)),
    _metrics_block(preds_04_cmp,       '04_Tuned_XGB',    len(SELECTED_FEATS)),
]
model_comparison_tuned = pd.DataFrame(rows)
print(model_comparison_tuned[['model', 'logloss_mean', 'logloss_std', 'n_matches']].to_string(index=False))

Comparison set: 6,862 matches
          model  logloss_mean  logloss_std  n_matches
       Baseline      1.848942     0.014083       6862
03_Selected_XGB      1.848385     0.005837       6862
   04_Tuned_XGB      1.840344     0.010204       6862


## 10. Save Outputs

In [19]:
final_logloss = float(model_comparison_tuned.loc[
    model_comparison_tuned['model'] == '04_Tuned_XGB', 'logloss_mean'].iloc[0])
final_std = float(model_comparison_tuned.loc[
    model_comparison_tuned['model'] == '04_Tuned_XGB', 'logloss_std'].iloc[0])

tuning_best = {
    'selected_features':        SELECTED_FEATS,
    'best_L':                   int(best_L),
    'best_ALPHA':               float(best_ALPHA),
    'best_xgb_params':          best_xgb_params,
    'final_validation_logloss': final_logloss,
    'final_validation_std':     final_std,
    'comparison_fixture_count': int(len(common)),
}
(TUNE_DIR / 'tuning_best.json').write_text(json.dumps(tuning_best, indent=2))

model_comparison_tuned.to_csv(TUNE_DIR / 'model_comparison_tuned.csv', index=False)

tuned_out = preds_04_xgb[[
    'match_id', 'val_season', 'league', 'total_goals', 'lambda_home', 'lambda_away', 'expected'
]].copy()
tuned_out.rename(columns={'val_season': 'season'}, inplace=True)
tuned_out['pmf_json'] = preds_04_xgb['pmf'].apply(lambda a: json.dumps(list(map(float, a))))
tuned_out.to_csv(TUNE_DIR / 'tuned_predictions.csv', index=False)

run_config_tuning = {
    'selected_features_source': '../Outputs/Features/selected_features.json',
    'selected_features':        SELECTED_FEATS,
    'starting_L':               int(L03),
    'starting_ALPHA':           float(ALPHA03),
    'best_L':                   int(best_L),
    'best_ALPHA':               float(best_ALPHA),
    'best_xgb_params':          best_xgb_params,
    'validation_seasons':       [str(s) for s in val_seasons],
    'test_season_cutoff':       TEST_SEASON,
    'search_settings': {
        'L_GRID':      L_GRID,
        'ALPHA_GRID':  ALPHA_GRID,
        'PAIR_BLOCKS': [[name, [[k, list(vs)] for k, vs in axes]] for name, axes in PAIR_BLOCKS],
        'XGB_STARTS':  ['03_best'],
        'RANDOM_SEED': RANDOM_SEED,
    },
    'stability_recheck': {
        'moved':               stability_moved,
        'best_L_recheck':      int(best_L_recheck),
        'best_ALPHA_recheck':  float(best_ALPHA_recheck),
    },
    'comparison_fixture_count': int(len(common)),
    'timestamp': datetime.datetime.now(datetime.UTC).isoformat(),
}
(TUNE_DIR / 'run_config_tuning.json').write_text(json.dumps(run_config_tuning, indent=2))

print(f'Saved \u2192 {TUNE_DIR}')
print(f'  tuning_best.json')
print(f'  run_config_tuning.json')
print(f'  model_comparison_tuned.csv')
print(f'  tuned_predictions.csv')
print(f'\nFinal tuned logloss : {final_logloss:.6f} (\u00b1{final_std:.6f})')
print(f'03 reference        : {NB03_LOGLOSS:.6f}')
print(f'Improvement         : {NB03_LOGLOSS - final_logloss:+.6f}')

Saved → ../Outputs/Tuning
  tuning_best.json
  run_config_tuning.json
  model_comparison_tuned.csv
  tuned_predictions.csv

Final tuned logloss : 1.840344 (±0.010204)
03 reference        : 1.848385
Improvement         : +0.008041
